# Relay extension of Oriane's analysis
This is a separate extension, not part of her original notebook scope. It uses
the same regression/decomposition and species-quadrature definitions. Cohorts
remain separate; wheat–maize is identified as relay from the actual calendar,
irrespective of the misleading DOE `same` field. No reference-LAI substitution
is used in the error calculations.


In [ ]:
from __future__ import annotations
import json, math, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "1-simul_stics").is_dir())
sys.path.insert(0, str(REPO / '4-analyses/1-code'))
from oriane_input_adapter import load_data
from oriane_analysis_metrics import compute_decomposition, compare_alternate_mixed
NORMALIZATION_MODE = 'original_095'
NORMALIZATION_FACTOR = 0.95
OUTPUT_DIR = REPO / '4-analyses/2-outputs/oriane_notebook_rerun_2026-11-10/relay_extension_original_095'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA = load_data(normalization_factor=NORMALIZATION_FACTOR)
np.random.seed(42)
sns.set_theme(style='whitegrid', font_scale=.85)
plt.rcParams.update({'figure.dpi':100, 'savefig.dpi':200})
def publish_figures(cell, name=None):
    for j, num in enumerate(plt.get_fignums()):
        fig = plt.figure(num)
        for ax in fig.axes:
            for getter, setter in ((ax.get_xlabel, ax.set_xlabel), (ax.get_ylabel, ax.set_ylabel)):
                label = getter()
                label = label.replace('lai_n', 'Reference LAI (laimax)')
                label = label.replace('laimax', 'Reference LAI (laimax)') if label == 'laimax' else label
                label = label.replace('DAE', 'Days since first retained tropical date')
                label = label.replace('faPAR STICS', f'STICS RAINT / (0.48 × {NORMALIZATION_FACTOR:g} × SW)')
                label = 'Archimed absorbed PAR / incident PAR' if label in ('faPAR','sum_fapar') else label
                label = label.replace('density', 'Nominal design density')
                setter(label)
            if ax.get_xlabel() == 'Date':
                ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=3,maxticks=6))
                ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
            ax.tick_params(axis='both', labelsize=8)
        legends=list(fig.legends)+[ax.get_legend() for ax in fig.axes if ax.get_legend() is not None]
        for legend in legends:
            title=legend.get_title().get_text()
            if title=='density':legend.set_title('Nominal design density')
            if title=='% sorghum':legend.set_title('Sorghum row share (%)')
            for text in legend.get_texts():
                if text.get_text()=='density':text.set_text('Nominal design density')
                if text.get_text()=='% sorghum':text.set_text('Sorghum row share (%)')
        title=(f'Separate Oriane-method extension: {globals().get("cohort", "")}'+chr(10)+NORMALIZATION_MODE
               if cell>=100 else f'Oriane analysis · {NORMALIZATION_MODE} · source cell {cell}')
        fig.suptitle(title, y=.995, fontsize=11)
        fig.subplots_adjust(top=.88, bottom=.13, hspace=.42, wspace=.25)
        stem = Path(name).stem if name else f'cell_{cell:02d}'
        if j: stem += f'_{j}'
        png, pdf = OUTPUT_DIR / (stem+'.png'), OUTPUT_DIR / (stem+'.pdf')
        outside = 0
        limits = []
        for ax in fig.axes:
            x0,x1 = ax.get_xlim(); y0,y1 = ax.get_ylim(); limits.append([float(x0),float(x1),float(y0),float(y1)])
            for collection in ax.collections:
                offsets = np.asarray(collection.get_offsets(), dtype=float)
                if offsets.ndim == 2 and offsets.shape[1] == 2:
                    valid = np.isfinite(offsets).all(axis=1)
                    outside += int(np.sum(valid & ((offsets[:,0]<x0)|(offsets[:,0]>x1)|(offsets[:,1]<y0)|(offsets[:,1]>y1))))
        note=None
        if outside and cell in (40,41,42,43):
            note=fig.text(.5,.015,f'Authored zoom: {outside} points outside axes; full-range companion supplied.',ha='center',fontsize=8)
        fig.savefig(png, dpi=200, bbox_inches='tight'); fig.savefig(pdf, bbox_inches='tight')
        if outside and cell in (40,41,42,43):
            for ax in fig.axes:
                points=[np.asarray(c.get_offsets(),float) for c in ax.collections]
                points=[p[np.isfinite(p).all(axis=1)] for p in points if p.ndim==2 and p.shape[1]==2]
                points=[p for p in points if len(p)]
                if points:
                    p=np.concatenate(points); bound=max(float(p.max())*1.08,ax.get_xlim()[1],ax.get_ylim()[1])
                    ax.set(xlim=(0,bound),ylim=(0,bound))
            if note is not None:note.set_text('Full-range companion: all scatter points shown; original thresholds retained.')
            fig.savefig(OUTPUT_DIR/(stem+'_unclipped.png'),dpi=200,bbox_inches='tight')
            fig.savefig(OUTPUT_DIR/(stem+'_unclipped.pdf'),bbox_inches='tight')
        display(fig)
        plt.close(fig)


In [ ]:
full = DATA['full_paired'].copy()
full['usm_number'] = pd.to_numeric(full['usm']) if pd.api.types.is_numeric_dtype(full.usm) else full.usm.str.extract(r'(\d+)$',expand=False).astype(int)
full['usm'] = full.usm_number.map(lambda n:f'usm_{n}')
full['algo'] = full.algorithm
full['faPAR'] = full.fapar_archimed
full['faPAR STICS'] = full.raint/(.48*NORMALIZATION_FACTOR*full.trg_n)
full['Date'] = pd.to_datetime(full.date)
full['cohort'] = np.where(full.usm_number>=46,'wheat_maize_relay',np.where(full.sowing_date_latest_crop=='same','tropical_simultaneous','tropical_relay'))
keep=(full.raint!=0)&(full.laimax!=0)&(full.hauteur!=0)&full.incident_comparable
keep &= np.isfinite(full[['faPAR','faPAR STICS']]).all(axis=1)
frame=full[keep].copy()
assert not frame.duplicated(['usm','algo','species','Date']).any()
results=[]; species_tables=[]; aggregate_tables=[]
for n,(cohort,part) in enumerate(frame.groupby('cohort',observed=True)):
    crops=('wheat','maize') if cohort=='wheat_maize_relay' else ('sorghum','maize')
    result=compute_decomposition(part,expected_species=crops)
    sp=result.species_metrics.assign(cohort=cohort); ag=result.intercrop_metrics.assign(cohort=cohort)
    species_tables.append(sp);aggregate_tables.append(ag)
    dates=part.Date
    g=sns.relplot(data=part,x='faPAR',y='faPAR STICS',row='species',col='algo',hue='row_label' if cohort=='wheat_maize_relay' else 'design',kind='scatter',height=3.2,aspect=1.05,s=12,alpha=.6)
    for ax in g.axes.flat:
        ax.axline((0,0),slope=1,color='grey',linestyle='--');ax.set(xlim=(0,1.1),ylim=(0,1.1))
    publish_figures(100+n*2,cohort+'_fraction_scatter.png')
    table=result.plot_metrics.merge(part[['usm','row_label','row_orientation','design']].drop_duplicates(),on='usm',validate='many_to_one')
    g=sns.relplot(data=table,x='RMSEu / std_3D',y='RMSEs / std_3D',row='species',col='algo',hue='row_label' if cohort=='wheat_maize_relay' else 'design',style='row_orientation',kind='scatter',height=3.2,aspect=1.05,s=40)
    from matplotlib.patches import Wedge
    bound=max(1.6,float(table[['RMSEu / std_3D','RMSEs / std_3D']].max().max())*1.1)
    for ax in g.axes.flat:
        ax.axline((0,0),slope=1,color='grey',linestyle='--');ax.set(xlim=(0,bound),ylim=(0,bound))
        for radius in (.5,.6,.7):ax.add_patch(Wedge((0,0),radius,0,90,alpha=.12,color='grey'))
    publish_figures(101+n*2,cohort+'_decomposition.png')
    results.append(dict(cohort=cohort,usms=int(part.usm.nunique()),rows=len(part),species_series=len(sp),intercrops=len(ag),max_identity_residual=result.max_identity_residual))
species_out=pd.concat(species_tables,ignore_index=True);aggregate_out=pd.concat(aggregate_tables,ignore_index=True)
assert len(species_out)==252 and len(aggregate_out)==126
species_out.to_csv(OUTPUT_DIR/'all_cohort_species_metrics.csv',index=False)
aggregate_out.to_csv(OUTPUT_DIR/'all_cohort_intercrop_metrics.csv',index=False)
frame.to_csv(OUTPUT_DIR/'all_cohort_paired_dates.csv',index=False)
summary=aggregate_out.groupby(['cohort','algo'],observed=True).agg(cases=('usm','size'),mean_RMSE_quadrature=('RMSE_intercrop','mean'),mean_rRMSE_quadrature=('rRMSE_intercrop','mean'))
summary.to_csv(OUTPUT_DIR/'cohort_summary.csv')
display(summary)
